In [109]:
import json

TRAM_TRAIN_PATH = "/home/simonettos/thijs/classification/classification/datasets/tram_train.json"
AUGMENTED_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/combined_6th_tram_global.json"
OUTPUT_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/combined_6th_tram_global_filtered.json"

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def extract_label_set(data):
    allowed = set()
    for labels in data.get("labels", {}).values():
        if isinstance(labels, list):
            for label in labels:
                if label:
                    allowed.add(str(label).strip())
    return allowed

def filter_augmented(data, allowed_labels):
    new_sentences = {}
    new_labels = {}

    sentence_map = data.get("sentence", {})
    labels_map = data.get("labels", {})

    for sent_id, sentence in sentence_map.items():
        labels = labels_map.get(sent_id, [])
        kept = [str(label).strip() for label in labels if str(label).strip() in allowed_labels]

        if kept:
            new_sentences[sent_id] = sentence
            new_labels[sent_id] = kept

    return {"sentence": new_sentences, "labels": new_labels}

def main():
    tram_data = load_json(TRAM_TRAIN_PATH)
    aug_data = load_json(AUGMENTED_PATH)

    allowed_labels = extract_label_set(tram_data)
    filtered_data = filter_augmented(aug_data, allowed_labels)

    save_json(filtered_data, OUTPUT_PATH)

    print(f"Allowed labels: {len(allowed_labels)}")
    print(f"Original augmented sentences: {len(aug_data.get('sentence', {}))}")
    print(f"Filtered augmented sentences: {len(filtered_data.get('sentence', {}))}")

if __name__ == "__main__":
    main()

Allowed labels: 50
Original augmented sentences: 2315
Filtered augmented sentences: 255


In [112]:
import json
import random
import csv
from pathlib import Path
from collections import defaultdict

# ============================================================
# CONFIG
# ============================================================

RECOVERED_JSON_PATH = "/home/simonettos/thijs/classification/classification/datasets/tram_augmented_embeddings.json"
TTP_DESC_JSON_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/mitre/ttp_descriptions2.json"

OUTPUT_JSON_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/audit_sample_embeddings.json"
OUTPUT_CSV_PATH = "/home/simonettos/thijs/data_augmentatio_stefano/audit_sample.csv"

RECOVERY_TYPE = "embeddings"   # "hierarchy", "semantic", or "pseudo_label"
TARGET_N = 50
RANDOM_SEED = 42

# If your recovered json only contains final labels, keep False.
# If it contains both original and recovered labels, set True and adapt the loader below.
HAS_SEPARATE_ORIGINAL_AND_RECOVERED = False


# ============================================================
# HELPERS
# ============================================================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def clean_text(x):
    if x is None:
        return ""
    return " ".join(str(x).split())

def short_description(desc, max_len=180):
    desc = clean_text(desc)
    if len(desc) <= max_len:
        return desc
    return desc[: max_len - 3].rstrip() + "..."

def derive_title_from_description(desc):
    """
    The attached file provides TTP descriptions, not guaranteed official titles.
    So this extracts a short first-clause preview as a reviewer aid.
    It is NOT guaranteed to be the official ATT&CK title.
    """
    desc = clean_text(desc)
    if not desc:
        return ""
    for sep in [".", ";", ":", "  "]:
        if sep in desc:
            first = desc.split(sep)[0].strip()
            if len(first) > 6:
                return first
    return desc[:100].strip()

def expand_pairs_from_nested_json(data, recovery_type):
    """
    Expected input shape like:
    {
      "sentence": {"0": "...", "1": "..."},
      "labels": {"0": ["T1001.001", "T1132.001"], "1": ["T1001.003"]}
    }

    Since this format does not include original_label separately,
    we set original_label = recovered_label by default.
    """
    rows = []
    sentence_map = data.get("sentence", {})
    labels_map = data.get("labels", {})

    for sent_id, sentence in sentence_map.items():
        sentence = clean_text(sentence)
        labels = labels_map.get(sent_id, [])
        if not isinstance(labels, list):
            continue

        for recovered_label in labels:
            recovered_label = clean_text(recovered_label)
            if not recovered_label:
                continue

            rows.append({
                "sentence_id": str(sent_id),
                "sentence": sentence,
                "original_label": recovered_label,
                "recovered_label": recovered_label,
                "recovery_type": recovery_type,
            })
    return rows

def deduplicate_pairs(rows):
    seen = set()
    deduped = []
    for r in rows:
        key = (
            r["sentence"].strip(),
            r["original_label"].strip(),
            r["recovered_label"].strip(),
            r["recovery_type"].strip(),
        )
        if key not in seen:
            seen.add(key)
            deduped.append(r)
    return deduped

def stratified_sample(rows, target_n=50, seed=42):
    """
    Light stratification by recovered label:
    1) try to take one random example per label
    2) fill the remaining budget randomly from leftovers

    This avoids over-representing very frequent techniques.
    """
    rng = random.Random(seed)

    by_label = defaultdict(list)
    for r in rows:
        by_label[r["recovered_label"]].append(r)

    for label in by_label:
        rng.shuffle(by_label[label])

    labels = list(by_label.keys())
    rng.shuffle(labels)

    sample = []
    used_ids = set()

    # First pass: up to 1 per label
    for label in labels:
        if len(sample) >= target_n:
            break
        candidates = by_label[label]
        for item in candidates:
            key = (item["sentence"], item["recovered_label"], item["recovery_type"])
            if key not in used_ids:
                sample.append(item)
                used_ids.add(key)
                break

    # Second pass: fill randomly from all leftovers
    if len(sample) < target_n:
        leftovers = []
        for label in labels:
            for item in by_label[label]:
                key = (item["sentence"], item["recovered_label"], item["recovery_type"])
                if key not in used_ids:
                    leftovers.append(item)

        rng.shuffle(leftovers)
        need = target_n - len(sample)
        sample.extend(leftovers[:need])

    rng.shuffle(sample)
    return sample

def attach_ttp_context(rows, ttp_desc_map):
    enriched = []
    for i, r in enumerate(rows, start=1):
        orig = r["original_label"]
        rec = r["recovered_label"]

        orig_desc = clean_text(ttp_desc_map.get(orig, ""))
        rec_desc = clean_text(ttp_desc_map.get(rec, ""))

        # NOTE:
        # The attached file appears to provide ATT&CK descriptions, not guaranteed official titles.
        # So these "title" fields are heuristic previews, not official ATT&CK names.
        orig_title_guess = derive_title_from_description(orig_desc)
        rec_title_guess = derive_title_from_description(rec_desc)

        enriched.append({
            "id": f"{r['recovery_type'][:4]}_{i:04d}",
            "recovery_type": r["recovery_type"],
            "sentence_id": r["sentence_id"],
            "sentence": r["sentence"],

            "original_label": orig,
            # "original_label_title_guess": orig_title_guess,
            # "original_label_description_preview": short_description(orig_desc),
            "original_label_description_full": orig_desc,

            # "recovered_label": rec,
            # "recovered_label_title_guess": rec_title_guess,
            # "recovered_label_description_preview": short_description(rec_desc),
            # "recovered_label_description_full": rec_desc,

            # Reviewer annotation fields
            "expert_judgment": "",
            "allowed_judgments": ["plausible", "mild", "negative"],
            # "failure_mode": "",
            "notes": ""
        })
    return enriched

def write_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def write_csv(data, path):
    if not data:
        return
    fieldnames = list(data[0].keys())
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)


# ============================================================
# MAIN
# ============================================================

def main():
    recovered_data = load_json(RECOVERED_JSON_PATH)
    ttp_desc_map = load_json(TTP_DESC_JSON_PATH)

    if HAS_SEPARATE_ORIGINAL_AND_RECOVERED:
        raise NotImplementedError(
            "Adapt the loader for your format with separate original/recovered labels."
        )
    else:
        rows = expand_pairs_from_nested_json(recovered_data, RECOVERY_TYPE)

    rows = deduplicate_pairs(rows)
    sampled = stratified_sample(rows, target_n=TARGET_N, seed=RANDOM_SEED)
    audit_items = attach_ttp_context(sampled, ttp_desc_map)

    write_json(audit_items, OUTPUT_JSON_PATH)
    write_csv(audit_items, OUTPUT_CSV_PATH)

    print(f"Total unique pairs available: {len(rows)}")
    print(f"Sampled pairs: {len(sampled)}")
    print(f"Saved reviewer-friendly JSON to: {OUTPUT_JSON_PATH}")
    print(f"Saved reviewer-friendly CSV to: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    main()

Total unique pairs available: 218
Sampled pairs: 50
Saved reviewer-friendly JSON to: /home/simonettos/thijs/data_augmentatio_stefano/audit_sample_embeddings.json
Saved reviewer-friendly CSV to: /home/simonettos/thijs/data_augmentatio_stefano/audit_sample.csv


In [ ]:
import json

INPUT_FILE = "/home/simonettos/thijs/classification/classification/datasets/tram_augmented_hierarchy.json"
OUTPUT_FILE = "/home/simonettos/thijs/data_augmentatio_stefano/annotation_ready.json"
RECOVERY_TYPE = "hierarchy"   # or "semantic"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

items = []
counter = 1

for sentence_id, sentence_text in data["sentence"].items():
    labels = data["labels"].get(sentence_id, [])
    for label in labels:
        items.append({
            "id": f"{RECOVERY_TYPE[:4]}_{counter:04d}",
            "recovery_type": RECOVERY_TYPE,
            "sentence_id": sentence_id,
            "sentence": sentence_text,
            "original_label": label,
            "recovered_label": label,
            "original_label_name": "",
            "recovered_label_name": "",
            "expert_judgment": "",
            "allowed_judgments": ["plausible", "mild", "negative"],
            "failure_mode": "",
            "notes": ""
        })
        counter += 1

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(items, f, indent=2, ensure_ascii=False)

print(f"Saved {len(items)} items to {OUTPUT_FILE}")

Saved 2412 items to /home/simonettos/thijs/data_augmentatio_stefano/annotation_ready.json
